# Lecture 10. AERO 331 (Winter 2026)

*Instructor:* Dr. Amuthan Ramabathiran  
*Email:* aramabat@calpoly.edu   

In [1]:
import numpy as np
import libFRC_v1_1 as frc

## Question 1

Since $\sigma > 0$ and $\tau > 0$, $\sigma_{11} = \sigma/2 + \tau > 0$. This implies that $X_1 = \sigma_{1u}^T$. The value of $X_2$, however, depends on the sign of $\sigma_{22} = \sigma/2 - \tau$. Let us consider the two possibilities:

(i) $\sigma_{22} = \sigma/2 - \tau > 0 \; \Rightarrow \; X_2 = \sigma_{2u}^T$ :

In [4]:
X1 = 2e9 # Pa
X2 = 0.08e9 # Pa
sigma_12u = 0.12e9 # Pa

tau = 100e6 # Pa

a = 0.25/X2**2 + 0.25/sigma_12u**2
b = tau*(1/X1**2 - 1/X2**2)
c = tau**2 * (2/X1**2 + 1/X2**2) - 1.0

sigma_1 = (-b + np.sqrt(b**2 - 4*a*c))/(2*a)
sigma_2 = (-b - np.sqrt(b**2 - 4*a*c))/(2*a)

print(sigma_1, sigma_1/2 - tau)
print(sigma_2, sigma_2/2 - tau)

233384371.59471837 16692185.797359183
43095628.40528167 -78452185.79735917


Since `sigma_2` violates the condition $\sigma_{22} > 0$, it is inadmissible. So a valid candidate for the critical stress is given by $\sigma^{(i)} = 233.38 MPa$. 

(ii) $\sigma_{22} = \sigma/2 - \tau < 0 \; \Rightarrow \; X_2 = \sigma_{2u}^C$ :

In [6]:
X2 = 0.28e9 # Pa

a = 0.25/X2**2 + 0.25/sigma_12u**2
b = tau*(1/X1**2 - 1/X2**2)
c = tau**2 * (2/X1**2 + 1/X2**2) - 1.0

sigma_1 = (-b + np.sqrt(b**2 - 4*a*c))/(2*a)
sigma_2 = (-b - np.sqrt(b**2 - 4*a*c))/(2*a)

print(sigma_1, sigma_1/2 - tau)
print(sigma_2, sigma_2/2 - tau)

238122177.33189088 19061088.66594544
-177269763.53878745 -188634881.76939374


We see that in this case, `sigma_1` is not admissible, but `sigma_2` is. We thus get our second estimate for the maximum critical stress as $\sigma^{(ii)} = -177.27 MPa$.

Of the two, only $\sigma^{(i)} > 0$, which is required in the question. Thus the critical stress is $\sigma = \sigma^{(i)} = 233.38 MPa$. 

*Note:* If you get multiple admissible stress values, choose the lowest one.

## Question 2

In this case, the shell is made of a $2 mm$ thick symmetric laminate made of carbon/epoxy with the following laminate code: $[0, \pm 45, 90]_S$. Our goal is to compute the critical stress $\sigma$ as before.

This is a fairly lengthy calculation--the individual steps are presented below **for a given value of $\sigma$**:

(i) Convert external loads to the running load $\mathbf{N}$ and the moment resultant $\mathbf{M}$. 

(ii) Compute the $A, B, D$ matrices for the laminate.

    (a) Compute $Q$ for the laminate.
    (b) Compute $\bar{Q}$ for each ply in the laminate.
    (c) Compute $A, B, D$ using $\bar{Q}$ and ply coordinates (along the $z$-direction).

(iii) Solve for $\bar{\mathbf{\epsilon}}$ and $\bar{\mathbf{\kappa}}$ by solving the following equation:
$$
\begin{bmatrix}
\mathbf{N}\\
\mathbf{M}
\end{bmatrix}
=
\begin{bmatrix}
A & B\\
B & D
\end{bmatrix}
\begin{bmatrix}
\bar{\mathbf{\epsilon}}\\
\bar{\mathbf{\kappa}}
\end{bmatrix}.
$$

(iv) Compute midplane strains in each ply using 
$$
\mathbf{\epsilon}^{(k)}= \bar{\mathbf{\epsilon}} - \bar{z}_k \bar{\mathbf{\kappa}}.
$$
Here, $\bar{z}_k$ denotes the $z$-coordinate of the midplane of the $k^\text{th}$ ply:
$$
\bar{z}_k = \frac{1}{2}(z_{k-1} + z_k).
$$

(v) Compute midplane stress in each ply using
$$
\mathbf{\sigma}^{(k)} = \bar{Q}_k \mathbf{\epsilon}^{(k)}. 
$$

(vi) Transform the midplane stress from the $xy$ coordinate system, $\mathbf{\sigma}^{(k)}$ to the stress in the $1-2$ coordinate system, $\hat{\mathbf{\sigma}}^{(k)}$,  **for each ply**, using
$$
\hat{\mathbf{\sigma}}^{(k)} = T_{\sigma}^{(k)} \mathbf{\sigma}^{(k)}.
$$

(vii) Use these midplane stress values in the natural coordinates for each ply to compute the failure criterion for each ply.

Note that this entire calculation is for a chosen value of $\sigma$. You need to perform these six steps for different values of $\sigma$ and find the smallest value of $\sigma$ for which at least one of the plies fails. For this reason, these six steps are coded inside a function `compute_failure_stress` below. The function `Tsai_Hill_criterion` implements the failure criterion for a given stress state in the ply coordinate system. The function `Ts` computes the stress transformation matrix. 

In [9]:
tau = 100 # MPa
t = 2e-3 # m

sigma_1u_T = 2e3 # MPa
sigma_1u_C = 1.1e3 # MPa
sigma_2u_T = 0.08e3 # MPa
sigma_2u_C = 0.28e3 # MPa
sigma_12u = 0.12e3 # MPa


def Ts(theta):
    thetar = theta * np.pi / 180
    c = np.cos(thetar)
    s = np.sin(thetar)
    return np.array([
        [c*c, s*s, 2*c*s],
        [s*s, c*c, -2*c*s],
        [-c*s, c*s, c*c - s*s]
    ])

def Tsai_Hill_criterion(s11, s22, s12, Y1T, Y1C, Y2T, Y2C, Y12):
    X1 = Y1T
    if s11 < 0.0:
        X1 = Y1C

    X2 = Y2T
    if s22 < 0.0:
        X2 = Y2C

    f = (s11/X1)**2 + (s22/X2)**2 + (s12/Y12)**2 - s11*s22/(X1**2) - 1.0
    return f

def compute_failure_stress(sigma):
    # Step 1
    N = np.array([
        sigma * t,
        0.0,
        tau * t
    ])
    M = np.array([
        0.0,
        0.0,
        0.0
    ])
    
    # Step 2
    list_z = np.array([
        -t/2 + i*t/8 for i in range(5)
    ]) # only for the bottom half because laminate is symmetric

    ply_1 = frc.Ply(
        theta=0, 
        zL=-t/2, zH=-t/2 + t/8,
        E1=1.4e5,
        E2=1e4,
        G12=7e3,
        nu12=0.3
    ) 
    ply_2 = frc.Ply(
        theta=45, 
        zL=-t/2 + t/8, zH=-t/2 + 2*t/8,
        E1=1.4e5,
        E2=1e4,
        G12=7e3,
        nu12=0.3
    )
    ply_3 = frc.Ply(
        theta=-45, 
        zL=-t/2 + 2*t/8, zH=-t/2 + 3*t/8,
        E1=1.4e5,
        E2=1e4,
        G12=7e3,
        nu12=0.3
    )
    ply_4 = frc.Ply(
        theta=90, 
        zL=-t/2 + 3*t/8, zH=-t/2 + 4*t/8,
        E1=1.4e5,
        E2=1e4,
        G12=7e3,
        nu12=0.3
    )

    Qbar_1 = ply_1.Qbar
    Qbar_2 = ply_2.Qbar
    Qbar_3 = ply_3.Qbar
    Qbar_4 = ply_4.Qbar

    A = 2 * (Qbar_1 + Qbar_2 + Qbar_3 + Qbar_4) * t/8

    B = np.zeros_like(A) # because the laminate is symmetric

    D = 2 * (
        Qbar_1 * (list_z[1]**3 - list_z[0]**3) / 3
        +
        Qbar_2 * (list_z[2]**3 - list_z[1]**3) / 3
        +
        Qbar_3 * (list_z[3]**3 - list_z[2]**3) / 3
        +
        Qbar_4 * (list_z[4]**3 - list_z[3]**3) / 3
    )

    # Step 3
    epsilon_bar = np.linalg.inv(A) @ N 
    kappa_bar = np.linalg.inv(D) @ M # zero in this case since M = 0.

    # Step 4
    zbar = np.array([-t/2 + i*(t/8) + t/16 for i in range(4)]) # need to consider only 4 plies because of symmetry
    epsilon_1 = epsilon_bar - zbar[0] * kappa_bar
    epsilon_2 = epsilon_bar - zbar[1] * kappa_bar
    epsilon_3 = epsilon_bar - zbar[2] * kappa_bar
    epsilon_4 = epsilon_bar - zbar[3] * kappa_bar
    
    # Step 5
    sigma_1 = Qbar_1 @ epsilon_1 
    sigma_2 = Qbar_2 @ epsilon_2 
    sigma_3 = Qbar_3 @ epsilon_3
    sigma_4 = Qbar_4 @ epsilon_4

    # Step 6
    sigma_hat_1 = Ts(0) @ sigma_1
    sigma_hat_2 = Ts(45) @ sigma_2
    sigma_hat_3 = Ts(-45) @ sigma_3
    sigma_hat_4 = Ts(90) @ sigma_4

    # Step 7
    f1 = Tsai_Hill_criterion(*sigma_hat_1, sigma_1u_T, sigma_1u_C, sigma_2u_T, sigma_2u_C, sigma_12u)
    f2 = Tsai_Hill_criterion(*sigma_hat_2, sigma_1u_T, sigma_1u_C, sigma_2u_T, sigma_2u_C, sigma_12u)
    f3 = Tsai_Hill_criterion(*sigma_hat_3, sigma_1u_T, sigma_1u_C, sigma_2u_T, sigma_2u_C, sigma_12u)
    f4 = Tsai_Hill_criterion(*sigma_hat_4, sigma_1u_T, sigma_1u_C, sigma_2u_T, sigma_2u_C, sigma_12u)

    return f1, f2, f3, f4

Let us try out different value of $\sigma$ and check when at least one ply fails.

In [11]:
compute_failure_stress(sigma=100)

(-0.9114166731211977,
 -0.9380191518418985,
 -0.8396975145078573,
 -0.880011661690471)

In [12]:
compute_failure_stress(sigma=500)

(-0.5331517450837312,
 -0.29818023493754964,
 -0.01707209851571312,
 0.25197354068443634)

In [13]:
compute_failure_stress(sigma=450)

(-0.608016678757813,
 -0.4312132745331809,
 -0.17821595175352833,
 0.027934802714403073)

In [14]:
compute_failure_stress(sigma=440)

(-0.6220440031725358,
 -0.4559619231291486,
 -0.2085867630779321,
 -0.014042981873666927)

In [16]:
compute_failure_stress(sigma=445)

(-0.6150697435618447,
 -0.4436650138029632,
 -0.19347877238752853,
 0.006827995295120903)

The critical value of $\sigma$ is close to $445 MPa$. The $90^\circ$ ply fails in this case.